In [1]:
!pip install ijson==3.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 2.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import ijson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from glob import glob

# loading in raw xsens and insole data

In [4]:
import pandas as pd
import os

def load_files_to_dfs(main_folder_path):
    honda_dfs = []

    for root, dirs, files in os.walk(main_folder_path):
        if 'xsens.csv' in files and 'insoles.csv' in files and 'labels.csv' in files:
            df_xsens = pd.read_csv(os.path.join(root, 'xsens.csv'))
            df_insole = pd.read_csv(os.path.join(root, 'insoles.csv'))
            df_labels = pd.read_csv(os.path.join(root, 'labels.csv'))

            # Merge the DataFrames on common columns
            df = pd.merge(df_xsens, df_insole, on=['time', 'participant_id', 'task'])
            df = pd.merge(df, df_labels, on=['time', 'participant_id', 'task'])

            # Extract participant number from the folder path
            participant_num = os.path.basename(root)
            df['participant_num'] = participant_num
            honda_dfs.append(df)

            print(f"Successfully loaded data from folder: {participant_num}")

    return honda_dfs

# Usage example
main_folder_path = '/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/data_set'
dataframes = load_files_to_dfs(main_folder_path)

Successfully loaded data from folder: id15
Successfully loaded data from folder: id01
Successfully loaded data from folder: id12
Successfully loaded data from folder: id14
Successfully loaded data from folder: id24
Successfully loaded data from folder: id23
Successfully loaded data from folder: id08
Successfully loaded data from folder: id07
Successfully loaded data from folder: id25
Successfully loaded data from folder: id22
Successfully loaded data from folder: id19
Successfully loaded data from folder: id02
Successfully loaded data from folder: id04
Successfully loaded data from folder: id10
Successfully loaded data from folder: id16
Successfully loaded data from folder: id18
Successfully loaded data from folder: id17
Successfully loaded data from folder: id05
Successfully loaded data from folder: id13
Successfully loaded data from folder: id03
Successfully loaded data from folder: id08
Successfully loaded data from folder: id22
Successfully loaded data from folder: id23
Successfull

In [5]:
def merge_by_participant(honda_dfs):
    merged_dfs = {}
    for df in honda_dfs:
        participant_id = df['participant_id'].unique()[0]
        if participant_id not in merged_dfs:
            merged_dfs[participant_id] = df
        else:
            merged_dfs[participant_id] = pd.concat([merged_dfs[participant_id], df], ignore_index=True)
    return list(merged_dfs.values())

In [6]:
merged_dataframes = merge_by_participant(dataframes)

In [7]:
merged_dataframes[0]['task'].unique()

array(['B', 'A', 'C'], dtype=object)

In [8]:
merged_dataframes[0]['walk_mode'].unique()

array(['walk', 'slope_down', 'slope_up', 'stairs_up', 'stairs_down',
       'pavement_down', 'pavement_up'], dtype=object)

In [9]:
all_sensor_types = ['orientation','position','velocity','acceleration','angularVelocity','angularAcceleration','sensorFreeAcceleration','sensorMagneticField','sensorOrientation','jointAngle','jointAngleXZY', 'Left', 'Right', 'Right__raw', 'Left__raw', 'Left__norm', 'Right__norm']

all_sensor_locations = ['Pelvis', 'L5', 'L3', 'T12', 'T8', 'Neck', 'Head', 'RightShoulder', 'RightUpperArm', 'RightForeArm', 'RightHand', 'LeftShoulder', 'LeftUpperArm', 'LeftForeArm', 'LeftHand', 'RightUpperLeg', 'RightLowerLeg',
                    'RightFoot', 'RightToe', 'LeftUpperLeg', 'LeftLowerLeg', 'LeftFoot', 'LeftToe', 'Arch', 'Hallux', 'Heel_L', 'Heel_R', 'Met1', 'Met3', 'Met5', 'Toes']

axes = ['x', 'y', 'z', 'ql', 'qi', 'qj', 'qk']

all_surface_types = ['walk', 'slope_down', 'slope_up', 'stairs_up', 'stairs_down', 'pavement_down', 'pavement_up']

columns = ['walk_mode', 'time', 'participant_id', 'task', 'sensor_location', 'orientation_q1', 'orientation_qi', 'orientation_qj', 'orientation_qk',
           'position_x', 'position_y', 'position_z', 'velocity_x', 'velocity_y', 'velocity_z', 'acceleration_x', 'acceleration_y', 'acceleration_z',
           'angularVelocity_x', 'angularVelocity_y', 'angularVelocity_z', 'angularAcceleration_x', 'angularAcceleration_y', 'angularAcceleration_z',
           'sensorFreeAcceleration_x', 'sensorFreeAcceleration_y', 'sensorFreeAcceleration_z', 'sensorMagneticField_x', 'sensorMagneticField_y',
           'sensorMagneticField_z', 'sensorOrientation_q1', 'sensorOrientation_qi', 'sensorOrientation_qj', 'sensorOrientation_qk', 'jointAngle_x',
           'jointAngle_y', 'jointAngle_z', 'jointAngleXZY_x', 'jointAngleXZY_y', 'jointAngleXZY_z', 'Left', 'Right', 'Right__raw', 'Left__raw', 'Left__norm', 'Right__norm']


In [10]:
def process_trial_df(trial_df):

    df_list = []
    for sensor_location in all_sensor_locations:
        sensor_columns = [col for col in trial_df.columns if sensor_location in col]
        if sensor_columns:
            df_by_sensor = pd.DataFrame({
                "time": trial_df["time"],
                "participant_id": trial_df["participant_id"],
                "sensor_location": sensor_location,
                "task": trial_df["task"],
                "surface": trial_df["walk_mode"],
                "insoles_RightFoot_is_step": trial_df["insoles_RightFoot_is_step"],
                "insoles_LeftFoot_is_step": trial_df["insoles_LeftFoot_is_step"],
                "insoles_RightFoot_is_lifted": trial_df["insoles_RightFoot_is_lifted"],
                "insoles_LeftFoot_is_lifted": trial_df["insoles_LeftFoot_is_lifted"],
            })
            for col in sensor_columns:
                new_column_name = col.replace(sensor_location, '').strip('_')
                df_by_sensor[new_column_name] = trial_df[col]
            df_list.append(df_by_sensor)
    return df_list

In [11]:
# Assuming you have a list of DataFrames, `dataframes`, where each DataFrame represents a participant's data

for df in dataframes:
    processed_dfs = process_trial_df(df)

    # Do something with the processed DataFrames, e.g., save to CSV
    for processed_df in processed_dfs:
        participant_id = processed_df['participant_id'].unique()[0]
        sensor_location = processed_df['sensor_location'].unique()[0]
        processed_df.to_csv(f"/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/processed data/participant_{participant_id}_{sensor_location}.csv", index=False)